# Fine-tune Amazon Nova with Amazon Bedrock

This notebook demonstrates how to fine-tune Amazon Nova using Amazon Bedrock. The process includes:

1. Creating an IAM service role for the fine-tuning job
2. Configuring an Amazon Bedrock customization job for parameter-efficient supervised fine-tuning of Amazon Nova
3. Submitting the customization job
4. Waiting for the customization job to complete

## Requirements

- Amazon Bedrock access with appropriate permissions
- Training data uploaded to an Amazon S3 bucket in the required format. If you have followed our [02_create_custom_dataset_amazon_nova.ipynb](02_create_custom_dataset_amazon_nova.ipynb) notebook then you already have the dataset. For reference here is the relevant documentation: ([User Guide for Amazon Nova - Preparing data for fine-tuning Understanding models](https://docs.aws.amazon.com/nova/latest/userguide/fine-tune-prepare-data-understanding.html) and [User Guide for Amazon Nova - Complete request schema](https://docs.aws.amazon.com/nova/latest/userguide/complete-request-schema.html))

In [ ]:
%pip install --quiet -U boto3 botocore pathvalidate pandas 's3fs>=2025.9.0'
# s3fs is for pandas to read the training and validation metrics from Amazon S3 output bucket

We need to restart the kernel to use the updated packages:

In [ ]:
from IPython import get_ipython
get_ipython().kernel.do_shutdown(restart=True)

In [ ]:
import boto3 
from botocore.config import Config
import sys
import json
import time 
import concurrent.futures
import tqdm
import os
import uuid
import ipywidgets as widgets
from IPython.display import display
from pathvalidate import sanitize_filename
from utils.bedrock import get_fine_tunable_vision_models, wait_for, customization_job_status, load_and_plot_metric
import re
from utils.helpers import find_file_with_resource

In [ ]:
import sagemaker

In [ ]:
request_uuid = uuid.uuid4()

In [ ]:
region="us-east-1" # us-east-1 for Amazon Nova models
boto_session = boto3.Session(
    region_name=region
)
session = sagemaker.Session(boto_session=boto_session)
default_bucket_name = session.default_bucket() 
dataset_s3_prefix = "fatura2-train-data-amazon-nova" 
dataset_s3_uri = f"s3://{default_bucket_name}/{dataset_s3_prefix}/"

In [ ]:
account_id = session.account_id()

In [ ]:
dataset_s3_uri

In [ ]:
my_config = Config(
    region_name = region, 
    signature_version = 'v4',
    retries = {
        'max_attempts': 5,
        'mode': 'standard'
    })

bedrock = boto_session.client(service_name="bedrock", config=my_config)

In [ ]:
def get_model_arn_by_id_from(fine_tunable_models, model_id):
    return next((model["modelArn"] for model in fine_tunable_models if model.get("modelId","") == model_id)) 

In [ ]:
fine_tunable_models = get_fine_tunable_vision_models(bedrock)

In [ ]:
global model_id
global model_arn
global safe_model_id
model_id = "amazon.nova-2-lite-v1:0:256k"
model_arn = get_model_arn_by_id_from(fine_tunable_models, model_id)
safe_model_id = sanitize_filename(model_id)

In [ ]:
model_dropdown = widgets.Dropdown(
    options=[model["modelId"] for model in fine_tunable_models],
    value=model_id,
    description='Model:',
    style={'description_width': 'initial'},
    layout={'width': 'auto'}
)

# Create select button
select_button = widgets.Button(
    description='Select Model',
    style={'description_width': 'initial'},
    button_style='success',
    layout={'width': 'auto'}
)

# Create output widget for status messages
output = widgets.Output()

def on_select_button_click(b):
    global model_id
    global model_arn
    global safe_model_id
    with output:
        output.clear_output()
        model_id = model_dropdown.value
        model_arn = get_model_arn_by_id_from(fine_tunable_models, model_id)
        safe_model_id = sanitize_filename(model_id)
        print(f"Selected model: {model_id}")
        print(f"Model ARN: {model_arn}")
        print(f"Path safe model id: {model_id}")

select_button.on_click(on_select_button_click)

# Display widgets
display(widgets.VBox([
    widgets.HTML("<h3>Select Model to Fine-Tune</h3>"),
    model_dropdown,
    select_button,
    output
]))


In [ ]:
from datetime import datetime

now = datetime.now()
formatted_time = now.strftime('%Y-%m-%d-%H-%M-%S') + "-" + now.strftime('%f')[:3] # Output: 2025-05-07-14-43-xx-xxx

In [ ]:
## Specify input S3 bucket
input_s3_uri = os.path.join(dataset_s3_uri,"conversations_train_nova_format.jsonl")
validation_s3_uri = os.path.join(dataset_s3_uri,"conversations_dev_nova_format.jsonl")
output_s3_uri = f"s3://{default_bucket_name}/bedrock-finetune-output/{safe_model_id}-{formatted_time}/"

## Create the Amazon Bedrock Customization Service Role

Amazon Bedrock need to be able to assume a service role that gives it access to your training and validation data. Please review the permissions for least privilege. Here is the relevant documentation [Amazon Bedrock User Guide - Create a service role for model customization](https://docs.aws.amazon.com/bedrock/latest/userguide/model-customization-iam-role.html).

In [ ]:
# Create IAM client
iam_client = session.boto_session.client('iam')

# Define the role name
role_name = "AmazonBedrockFineTuneServiceRole"

# Define the policy document
policy_document = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Action": [
                "s3:GetObject",
                "s3:ListBucket"
            ],
            "Resource": [
                f"arn:aws:s3:::{default_bucket_name}",
                f"arn:aws:s3:::{default_bucket_name}/*",
            ]
        },
        {
            "Effect": "Allow",
            "Action": [
                "s3:PutObject",
            ],
            "Resource": [
                f"arn:aws:s3:::{default_bucket_name}/bedrock-finetune-output/*"
            ]
        }
    ]
}

# Define the trust relationship
trust_relationship = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Principal": {
                "Service": "bedrock.amazonaws.com"
            },
            "Action": "sts:AssumeRole",
            "Condition": {
                "StringEquals": {
                    "aws:SourceAccount": account_id
                },
                "ArnEquals": {
                    "aws:SourceArn": f"arn:aws:bedrock:{my_config.region_name}:{account_id}:model-customization-job/*"
                }
            }
        }
    ]
}



# Check if the role already exists
try:
    existing_role = iam_client.get_role(RoleName=role_name)
    print(f"Role '{role_name}' already exists. Skipping creation.")
    role_arn = existing_role['Role']['Arn']
except iam_client.exceptions.NoSuchEntityException:
    # Create the role
    try:
        create_role_response = iam_client.create_role(
            RoleName=role_name,
            AssumeRolePolicyDocument=json.dumps(trust_relationship),
            Description="Service role for Amazon Bedrock Fine-Tuning"
        )
        
        # Attach the inline policy to the role
        iam_client.put_role_policy(
            RoleName=role_name,
            PolicyName="AmazonBedrockCustomizationServiceRolePolicy",
            PolicyDocument=json.dumps(policy_document)
        )
        
        print(f"Successfully created role: {create_role_response['Role']['Arn']}")
        role_arn = create_role_response['Role']['Arn']
    except Exception as e:
        print(f"Error creating role: {str(e)}")



## Configure the Model Customization Job

In [ ]:
import re

CUSTOM_MODEL_NAME_MAX_LENGTH = 63

def create_model_name(dataset_prefix: str, formatted_time: str, max_length: int = 63) -> str:
    """
    Create a model name by combining dataset prefix and timestamp.
    
    Requirements:
    - Format: {prefix}-{timestamp}
    - Maximum length of 63 characters
    - No consecutive hyphens
    """
    # Clean consecutive hyphens from prefix
    clean_prefix = re.sub(r'-{2,}', '-', dataset_prefix).strip('-')
    
    # Calculate available space for prefix (accounting for separator and timestamp)
    max_prefix_length = max_length - len(formatted_time) - 1
    
    # Truncate prefix if needed and remove trailing hyphen from truncation
    truncated_prefix = clean_prefix[:max_prefix_length].rstrip('-')
    
    # Combine and return
    return f"{truncated_prefix}-{formatted_time}"


In [ ]:
job_name = create_model_name(dataset_s3_prefix, formatted_time, CUSTOM_MODEL_NAME_MAX_LENGTH)
model_name = job_name

In [ ]:
# Select the customization type from "FINE_TUNING" or "CONTINUED_PRE_TRAINING". 
customization_type = "FINE_TUNING"


# Define the hyperparameters for fine-tuning Amazon Nova model
hyper_parameters = {
    "epochCount": "4", # default: 2
    "learningRate": '0.00001', 
    "batchSize": "1",
    "learningRateWarmupSteps": '2' # Nova Lite: (dataset size / 160), Nova Pro: (dataset size / 320)
}


response_ft = bedrock.create_model_customization_job(
    customizationType=customization_type,
    clientRequestToken=str(request_uuid),
    jobName = job_name,
    customModelName = model_name,
    roleArn = role_arn,
    baseModelIdentifier = model_arn,
    hyperParameters=hyper_parameters,
    trainingDataConfig={
        "s3Uri": input_s3_uri
    },
    # Nova 2.0 doesn't support validation set
    # validationDataConfig={
    #     "validators": [
    #         {"s3Uri": validation_s3_uri},
    #     ]
    # },
    outputDataConfig={"s3Uri": output_s3_uri},
)


In [ ]:
response_ft

## Wait for fine-tuning job to complete

In [ ]:
jobArn = response_ft.get('jobArn')

In [ ]:
# wait for 1.5 hours for the training job to complete
wait_for(customization_job_status, max_wait_seconds=5400, check_interval=360, job_arn=jobArn, bedrock=bedrock)

## Optional Plot Training Metrics

Optionally, you can also plot training loss and validation loss using the step_wise_training_metrics.csv file generated from the finetuning job. This csv file and other model artifacts can be found under Amazon Bedrock -> Custom model -> Custom model name -> Output data (S3 location)

In [ ]:
response = bedrock.get_model_customization_job(
    jobIdentifier=jobArn
)

In [ ]:
if response['status'] != 'Completed':
    print('❌ the model customization job is not complete. Wait for it to finish or check if it failed.')

In [ ]:
s3_job_output_location = response['outputDataConfig']['s3Uri']

In [ ]:
load_and_plot_metric(s3_job_output_location,'step_wise_training_metrics.csv','training_loss') # optionally save to file 'training_loss.png')

In [ ]:
load_and_plot_metric(response['outputDataConfig']['s3Uri'],'validation_metrics.csv','validation_loss')